# NovaPay ML API — Pruebas de Endpoints - Local

Notebook para probar los 3 endpoints de la API en local.

**Requisitos antes de ejecutar:**
- API corriendo en `http://localhost:8000`
- Docker corriendo con `postgres-demo`
- BD `novapay` creada con las tablas

**Endpoints a probar:**
- `GET  /health`
- `POST /predict`
- `POST /predict/batch`
- `GET  /metrics`

## 0. Imports y configuración

In [3]:
import requests
import pandas as pd
import json

# URL base de la API
BASE_URL = 'http://localhost:8000'

print('Imports correctos')
print(f'URL de la API: {BASE_URL}')

Imports correctos
URL de la API: http://localhost:8000


## 1. GET /health — ¿Está viva la API?

In [4]:
respuesta = requests.get(f'{BASE_URL}/health')

print(f'Status code: {respuesta.status_code}')
print(f'Respuesta:')
print(json.dumps(respuesta.json(), indent=2))

Status code: 200
Respuesta:
{
  "status": "ok",
  "version": "4.0.0",
  "modelo": "modelo_07_v1"
}


## 2. POST /predict — Predecir UNA transacción

### 2a. Transacción legítima

In [5]:
# Transacción legítima — comportamiento normal
transaccion_legitima = {
    "id_transaccion"                        : "test-legitima-9999",
    "id_cliente"                            : "cliente-001",
    "tipo_cliente"                          : "persona",
    "edad_cliente"                          : 35,
    "customer_country"                      : "ES",
    "customer_region"                       : "Centro",
    "tenure"                                : 730,
    "importe_medio_mensual"                 : 800.00,
    "desviacion_estandar_mensual"           : 150.00,
    "media_transacciones_al_dia"            : 3.5,
    "numero_fraudes_ultimo_ano"             : 0,
    "id_cuenta"                             : "cuenta-001",
    "cuenta_origen"                         : "ES20427866183",
    "estado_cuenta"                         : "activa",
    "saldo_actual"                          : 2500.00,
    "saldo_medio_30_dias"                   : 2200.00,
    "volumen_entrante_30_dias"              : 3000.00,
    "volumen_saliente_30_dias"              : 2800.00,
    "numero_transferencias_recibidas_7_dias": 3,
    "numero_transferencias_enviadas_7_dias" : 2,
    "id_tarjeta"                            : "tarjeta-001",
    "estado_tarjeta"                        : "activa",
    "fecha_creacion_tarjeta"                : "2023-01-15",
    "antiguedad_tarjeta_dias"               : 490,
    "limite_importe_transacciones"          : 2000.00,
    "veces_superar_limite_7_dias"           : 0,
    "tipo_transaccion"                      : "tarjeta",
    "fecha_hora"                            : "2026-05-23 14:30:00",
    "is_night"                              : 0,
    "is_weekend"                            : 0,
    "tiempo_desde_ultima_transaccion"       : 2600,
    "numero_transacciones_ultima_hora"      : 1,
    "importe_transaccion"                   : 150.00,
    "metodo_autenticacion"                  : "PIN",
    "numero_pin_disponibles"                : 3,
    "identificador_dispositivo_fingerprint" : "fa1bdf50-1c95",
    "dispositivo_reconocido"                : 1,
    "operacion_pais"                        : "ES",
    "operacion_region"                      : "Centro",
    "direccion_ip_origen"                   : "86.34.12.179",
    "geolocalizacion"                       : "40.4168,-3.7038",
    "cuenta_destino"                        : "ES169540317577",
    "destino_alto_riesgo"                   : 0
}

respuesta = requests.post(
    f'{BASE_URL}/predict',
    json=transaccion_legitima
)

print(f'Status code: {respuesta.status_code}')
print(f'Respuesta:')
print(json.dumps(respuesta.json(), indent=2))

Status code: 200
Respuesta:
{
  "id_transaccion": "test-legitima-9999",
  "is_fraud": 0,
  "prob_fraud": 0.2506,
  "impacto_fraude": 0,
  "es_transfronteriza": 0,
  "ratio_imp_limite": 0.075,
  "intensidad_tx": 0.0004,
  "severidad_tx": 150.0,
  "flujo_neto_30d": 200.0,
  "mensaje": "Transaccion legitima - probabilidad fraude 25%"
}


### 2b. Transacción sospechosa — simulando ataque de Ciber

In [6]:
# Transacción sospechosa — Ciber simulando fraude
# Rompe varias reglas de negocio a la vez
transaccion_fraude = {
    "id_transaccion"                        : "test-fraude-2222",
    "id_cliente"                            : "cliente-002",
    "tipo_cliente"                          : "persona",
    "edad_cliente"                          : 25,
    "customer_country"                      : "ES",   # registrado en España
    "customer_region"                       : "Centro",
    "tenure"                                : 10,     # cuenta muy nueva
    "importe_medio_mensual"                 : 200.00,
    "desviacion_estandar_mensual"           : 50.00,
    "media_transacciones_al_dia"            : 1.0,
    "numero_fraudes_ultimo_ano"             : 2,      # ya tuvo fraudes
    "id_cuenta"                             : "cuenta-002",
    "cuenta_origen"                         : "ES20427866184",
    "estado_cuenta"                         : "activa",
    "saldo_actual"                          : 100.00,
    "saldo_medio_30_dias"                   : 150.00,
    "volumen_entrante_30_dias"              : 500.00,
    "volumen_saliente_30_dias"              : 5000.00, # sale mucho más de lo que entra
    "numero_transferencias_recibidas_7_dias": 1,
    "numero_transferencias_enviadas_7_dias" : 20,     # muchas transferencias
    "id_tarjeta"                            : "tarjeta-002",
    "estado_tarjeta"                        : "activa",
    "fecha_creacion_tarjeta"                : "2026-05-01",
    "antiguedad_tarjeta_dias"               : 5,      # tarjeta muy nueva
    "limite_importe_transacciones"          : 500.00,
    "veces_superar_limite_7_dias"           : 5,      # supera límite frecuentemente
    "tipo_transaccion"                      : "transferencia",
    "fecha_hora"                            : "2026-05-23 03:00:00", # madrugada
    "is_night"                              : 1,      # de noche
    "is_weekend"                            : 0,
    "tiempo_desde_ultima_transaccion"       : 30,     # hace solo 30 segundos
    "numero_transacciones_ultima_hora"      : 15,     # muchas en poco tiempo
    "importe_transaccion"                   : 4500.00, # importe enorme
    "metodo_autenticacion"                  : "firma",
    "numero_pin_disponibles"                : 0,      # sin intentos PIN
    "identificador_dispositivo_fingerprint" : None,
    "dispositivo_reconocido"                : 0,      # dispositivo desconocido
    "operacion_pais"                        : "RU",   # operación desde Rusia
    "operacion_region"                      : "Moscú",
    "direccion_ip_origen"                   : "185.220.101.45",
    "geolocalizacion"                       : "55.7558,37.6173",
    "cuenta_destino"                        : "ES999999999999",
    "destino_alto_riesgo"                   : 1       # cuenta destino de riesgo
}

respuesta = requests.post(
    f'{BASE_URL}/predict',
    json=transaccion_fraude
)

print(f'Status code: {respuesta.status_code}')
print(f'Respuesta:')
print(json.dumps(respuesta.json(), indent=2))

Status code: 200
Respuesta:
{
  "id_transaccion": "test-fraude-2222",
  "is_fraud": 1,
  "prob_fraud": 0.6526,
  "impacto_fraude": 3,
  "es_transfronteriza": 1,
  "ratio_imp_limite": 9.0,
  "intensidad_tx": 0.4839,
  "severidad_tx": 67500.0,
  "flujo_neto_30d": -4500.0,
  "mensaje": "FRAUDE DETECTADO - probabilidad 65%"
}


## 3. POST /predict/batch — Predecir VARIAS transacciones

### 3a. Batch manual — mezcla de legítimas y fraudulentas

In [7]:
# Lista con 1 transacciones: 1 legítimas y 1 sospechosa
batch = [
    {
        "id_transaccion"                        : "batch-001",
        "id_cliente"                            : "cliente-001",
        "tipo_cliente"                          : "persona",
        "edad_cliente"                          : 35,
        "customer_country"                      : "ES",
        "customer_region"                       : "Centro",
        "tenure"                                : 730,
        "importe_medio_mensual"                 : 500.00,
        "desviacion_estandar_mensual"           : 150.00,
        "media_transacciones_al_dia"            : 3.5,
        "numero_fraudes_ultimo_ano"             : 0,
        "id_cuenta"                             : "cuenta-001",
        "cuenta_origen"                         : "ES20427866183",
        "estado_cuenta"                         : "activa",
        "saldo_actual"                          : 2500.00,
        "saldo_medio_30_dias"                   : 2200.00,
        "volumen_entrante_30_dias"              : 3000.00,
        "volumen_saliente_30_dias"              : 2800.00,
        "numero_transferencias_recibidas_7_dias": 3,
        "numero_transferencias_enviadas_7_dias" : 2,
        "id_tarjeta"                            : "tarjeta-001",
        "estado_tarjeta"                        : "activa",
        "fecha_creacion_tarjeta"                : "2023-01-15",
        "antiguedad_tarjeta_dias"               : 490,
        "limite_importe_transacciones"          : 2000.00,
        "veces_superar_limite_7_dias"           : 0,
        "tipo_transaccion"                      : "tarjeta",
        "fecha_hora"                            : "2026-05-23 10:00:00",
        "is_night"                              : 0,
        "is_weekend"                            : 0,
        "tiempo_desde_ultima_transaccion"       : 7200,
        "numero_transacciones_ultima_hora"      : 1,
        "importe_transaccion"                   : 80.00,
        "metodo_autenticacion"                  : "PIN",
        "numero_pin_disponibles"                : 3,
        "identificador_dispositivo_fingerprint" : "dispositivo-001",
        "dispositivo_reconocido"                : 1,
        "operacion_pais"                        : "ES",
        "operacion_region"                      : "Centro",
        "direccion_ip_origen"                   : "86.34.12.179",
        "geolocalizacion"                       : "40.4168,-3.7038",
        "cuenta_destino"                        : "ES169540317577",
        "destino_alto_riesgo"                   : 0
    },
    {
        "id_transaccion"                        : "batch-002",
        "id_cliente"                            : "cliente-002",
        "tipo_cliente"                          : "persona",
        "edad_cliente"                          : 25,
        "customer_country"                      : "ES",
        "customer_region"                       : "Centro",
        "tenure"                                : 10,
        "importe_medio_mensual"                 : 200.00,
        "desviacion_estandar_mensual"           : 50.00,
        "media_transacciones_al_dia"            : 1.0,
        "numero_fraudes_ultimo_ano"             : 2,
        "id_cuenta"                             : "cuenta-002",
        "cuenta_origen"                         : "ES20427866184",
        "estado_cuenta"                         : "activa",
        "saldo_actual"                          : 100.00,
        "saldo_medio_30_dias"                   : 150.00,
        "volumen_entrante_30_dias"              : 500.00,
        "volumen_saliente_30_dias"              : 5000.00,
        "numero_transferencias_recibidas_7_dias": 1,
        "numero_transferencias_enviadas_7_dias" : 20,
        "id_tarjeta"                            : "tarjeta-002",
        "estado_tarjeta"                        : "activa",
        "fecha_creacion_tarjeta"                : "2026-05-01",
        "antiguedad_tarjeta_dias"               : 5,
        "limite_importe_transacciones"          : 500.00,
        "veces_superar_limite_7_dias"           : 5,
        "tipo_transaccion"                      : "transferencia",
        "fecha_hora"                            : "2026-05-23 03:00:00",
        "is_night"                              : 1,
        "is_weekend"                            : 0,
        "tiempo_desde_ultima_transaccion"       : 30,
        "numero_transacciones_ultima_hora"      : 15,
        "importe_transaccion"                   : 4500.00,
        "metodo_autenticacion"                  : "firma",
        "numero_pin_disponibles"                : 0,
        "identificador_dispositivo_fingerprint" : None,
        "dispositivo_reconocido"                : 0,
        "operacion_pais"                        : "RU",
        "operacion_region"                      : "Moscu",
        "direccion_ip_origen"                   : "185.220.101.45",
        "geolocalizacion"                       : "55.7558,37.6173",
        "cuenta_destino"                        : "ES999999999999",
        "destino_alto_riesgo"                   : 1
    }
]

respuesta = requests.post(
    f'{BASE_URL}/predict/batch',
    json=batch
)

resultado = respuesta.json()
print(f'Status code: {respuesta.status_code}')
print(f'Total enviadas: {resultado["total"]}')
print(f'Fraudes:        {resultado["fraudes"]}')
print(f'Legitimas:      {resultado["legitimas"]}')
print()

for pred in resultado['predicciones']:
    print(f'  {pred["id_transaccion"]:20s} → is_fraud: {pred["is_fraud"]} | prob: {pred["prob_fraud"]} | {pred["mensaje"]}')

Status code: 200
Total enviadas: 2
Fraudes:        1
Legitimas:      1

  batch-001            → is_fraud: 0 | prob: 0.238 | Transaccion legitima - probabilidad fraude 24%
  batch-002            → is_fraud: 1 | prob: 0.6526 | FRAUDE DETECTADO - probabilidad 65%


### 3b. Batch desde el CSV - primeras 10 filas

In [8]:
# Cargar 10 filas del CSV y mandarlas en batch
df = pd.read_csv('dataset_fraude.csv')
muestra = df.iloc[:10]

batch_csv = []
for _, fila in muestra.iterrows():
    batch_csv.append({
        "id_transaccion"                        : fila['id_transaccion'] + '-batch',
        "id_cliente"                            : fila['id_cliente'],
        "tipo_cliente"                          : fila['tipo_cliente'],
        "edad_cliente"                          : int(fila['edad_cliente']),
        "customer_country"                      : fila['customer_country'],
        "customer_region"                       : fila['customer_region'],
        "tenure"                                : int(fila['tenure']),
        "importe_medio_mensual"                 : float(fila['importe_medio_mensual']),
        "desviacion_estandar_mensual"           : float(fila['desviacion_estandar_mensual']),
        "media_transacciones_al_dia"            : float(fila['media_transacciones_al_dia']),
        "numero_fraudes_ultimo_ano"             : int(fila['numero_fraudes_ultimo_ano']),
        "id_cuenta"                             : fila['id_cuenta'],
        "cuenta_origen"                         : fila['cuenta_origen'],
        "estado_cuenta"                         : fila['estado_cuenta'],
        "saldo_actual"                          : float(fila['saldo_actual']),
        "saldo_medio_30_dias"                   : float(fila['saldo_medio_30_dias']),
        "volumen_entrante_30_dias"              : float(fila['volumen_entrante_30_dias']),
        "volumen_saliente_30_dias"              : float(fila['volumen_saliente_30_dias']),
        "numero_transferencias_recibidas_7_dias": int(fila['numero_transferencias_recibidas_7_dias']),
        "numero_transferencias_enviadas_7_dias" : int(fila['numero_transferencias_enviadas_7_dias']),
        "id_tarjeta"                            : fila['id_tarjeta'],
        "estado_tarjeta"                        : fila['estado_tarjeta'],
        "fecha_creacion_tarjeta"                : str(fila['fecha_creacion_tarjeta']),
        "antiguedad_tarjeta_dias"               : int(fila['antiguedad_tarjeta_dias']),
        "limite_importe_transacciones"          : float(fila['limite_importe_transacciones']),
        "veces_superar_limite_7_dias"           : int(fila['veces_superar_limite_7_dias']),
        "tipo_transaccion"                      : fila['tipo_transaccion'],
        "fecha_hora"                            : str(fila['fecha_hora']),
        "is_night"                              : int(fila['is_night']),
        "is_weekend"                            : int(fila['is_weekend']),
        "tiempo_desde_ultima_transaccion"       : int(fila['tiempo_desde_ultima_transaccion']),
        "numero_transacciones_ultima_hora"      : int(fila['numero_transacciones_ultima_hora']),
        "importe_transaccion"                   : float(fila['importe_transaccion']),
        "metodo_autenticacion"                  : fila['metodo_autenticacion'],
        "numero_pin_disponibles"                : int(fila['numero_pin_disponibles']),
        "identificador_dispositivo_fingerprint" : fila['identificador_dispositivo_fingerprint'],
        "dispositivo_reconocido"                : int(fila['dispositivo_reconocido']),
        "operacion_pais"                        : fila['operacion_pais'],
        "operacion_region"                      : fila['operacion_region'],
        "direccion_ip_origen"                   : fila['direccion_ip_origen'],
        "geolocalizacion"                       : fila['geolocalizacion'],
        "cuenta_destino"                        : fila['cuenta_destino'],
        "destino_alto_riesgo"                   : int(fila['destino_alto_riesgo'])
    })

respuesta = requests.post(
    f'{BASE_URL}/predict/batch',
    json=batch_csv
)

resultado = respuesta.json()
print(f'Status code: {respuesta.status_code}')
print(f'Total enviadas: {resultado["total"]}')
print(f'Fraudes:        {resultado["fraudes"]}')
print(f'Legitimas:      {resultado["legitimas"]}')
print()

# Comparar predicciones con valores reales del CSV
print('Comparativa modelo vs CSV:')
print(f'{"ID":25s} {"Real":6s} {"Pred":6s} {"Prob":6s} {"OK?"}')
print('-' * 60)
for i, pred in enumerate(resultado['predicciones']):
    real = int(muestra.iloc[i]['IS_FRAUD'])
    ok   = '✅' if real == pred['is_fraud'] else '❌'
    print(f'{pred["id_transaccion"]:25s} {real:6d} {pred["is_fraud"]:6d} {pred["prob_fraud"]:6.4f} {ok}')

Status code: 200
Total enviadas: 10
Fraudes:        6
Legitimas:      4

Comparativa modelo vs CSV:
ID                        Real   Pred   Prob   OK?
------------------------------------------------------------
059638c5-40f-batch             0      1 0.3467 ❌
a00ed476-b71-batch             0      1 0.4248 ❌
53673521-1b0-batch             0      0 0.1466 ✅
a8a58fe8-940-batch             0      0 0.2755 ✅
e99741de-240-batch             0      0 0.1875 ✅
7b5634e7-610-batch             0      1 0.6057 ❌
cbccff1e-441-batch             0      0 0.2395 ✅
8517613c-de0-batch             1      1 0.7419 ✅
dd5ac3ec-10e-batch             0      1 0.3911 ❌
9f5c2495-5f3-batch             0      1 0.7188 ❌


## 4. GET /metrics — Métricas del modelo (END-POINT eliminado)

In [11]:

#respuesta = requests.get(f'{BASE_URL}/metrics')

#print(f'Status code: {respuesta.status_code}')
#print(f'Metricas:')
#print(json.dumps(respuesta.json(), indent=2))
#'''